# generate_retailmax_data.ipynb

Genera los datos sintéticos del sistema transaccional de **RetailMax**
(Escenario B — Retail y Comercio Electrónico), cubriendo las entidades
necesarias para el modelamiento dimensional y los procesos de ML/análisis
posteriores. Produce las siguientes tablas exigidas por la prueba técnica:

- `MSTR_PROVEEDORES`
- `MSTR_ARTICULOS`
- `MSTR_TIENDAS`
- `CRM_MIEMBROS`
- `TRANS_VENTAS`
- `INV_STOCK_DIARIO`
- `POST_DEVOLUCIONES`

## Características

- **Reproducible** mediante semilla aleatoria fija (`config.yaml`)
- **Distribuciones realistas**: horarios pico, edades normales, Pareto en ventas
- **Integridad referencial** entre tablas de hechos y dimensiones
- **~5% de nulos controlados** en campos no críticos
- **Cobertura temporal** ≥ 12 meses
- **Anomalías intencionales documentadas** (ver [`docs/anomalias.md`](docs/anomalias.md))
- **Salida en múltiples formatos**: CSV, JSON

## Uso

```bash
python generate_data.py --config config.yaml

# Para pruebas rápidas (muestra reducida)
python generate_data.py --config config.yaml --sample 0.01
```

## Instalar las librerías

Este proyecto utiliza las siguientes librerías para la generación de datos sintéticos de RetailMax:

- **Faker**: genera datos sintéticos realistas (nombres, direcciones, correos, fechas, etc.) en las tablas dimensionales como `MSTR_ARTICULOS`, `CRM_MIEMBROS` y `MSTR_TIENDAS`.
- **PyYAML**: lee el archivo de configuración `config.yaml` (semilla, volúmenes, rango de fechas, formatos de salida).
- **pandas**: estructura y exporta los datos generados a los distintos formatos (CSV, JSON).
- **numpy**: genera las distribuciones numéricas (edades normales, precios lognormal, horarios pico, etc.) de forma vectorizada y reproducible.

In [37]:
%pip install faker
%pip install PyYAML
%pip install pandas
%pip install numpy
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importar las librerías

Se importan las librerías necesarias para la generación de datos sintéticos: `Faker` para datos realistas, `yaml` para leer la configuración, `pandas`/`numpy` para estructurar y generar las distribuciones de datos, y las utilidades estándar (`argparse`, `json`, `os`, `random`, `time`, `datetime`) para el manejo de parámetros, formatos de salida, semillas de reproducibilidad y fechas.

In [38]:
from faker import Faker
import yaml
import pandas as pd
import numpy as np

import argparse
import json
import os
import random
import time
from datetime import datetime, timedelta
from scipy.stats import truncnorm
import uuid

### Cargar configuración y fijar semilla

In [39]:
# Abrimos el archivo de configuración y cargamos los datos
with open("config.yaml", "r", encoding="utf-8") as archivo:
    datos = yaml.full_load(archivo)

# Extraemos la semilla del archivo de configuración
semilla = datos["semilla"]

# Fijamos la semilla para que siempre devuelva los mismos datos
Faker.seed(semilla)
random.seed(semilla)
np.random.seed(semilla)

fake = Faker("es_CO")  # Configuramos Faker para generar datos en español (Colombia)

### Extracción de parámetros de configuración

Se cargan en variables individuales todos los parámetros definidos en el archivo de configuración (`datos`), organizados por tipo:

- **Distribuciones**: tipos de tienda, canales, macro categorías, rangos de precios por categoría, países, centros de distribución, stock, franjas horarias, días de la semana, edades, bins de rango de edad, género, tipos de pago, descuento aplicado, motivos y estados de devolución, calificación de proveedores, activos y anomalías.
- **Fechas**: rango de fechas a generar.
- **Volúmenes**: cantidad de registros a simular por entidad.
- **Otros**: porcentaje de valores nulos a inyectar y formatos de salida del dataset.

Estas variables se utilizarán posteriormente como parámetros de entrada para las funciones de generación de datos sintéticos.

In [40]:
# Distribuciones
tipos_tienda = datos["tipos_tienda"]
canales = datos["canales"]
macro_categorias = datos["macro_categorias"]
rango_precios_por_categoria = datos["rango_precios_por_categoria"]
paises = datos["paises"]
centros_distribucion = datos["centros_distribucion"]
stock = datos["stock"]
franjas_horarias = datos["franjas_horarias"]
dias_semana = datos["dias_semana"]
edades = datos["edades"]
rango_edad_bins = datos["rango_edad_bins"]
genero = datos["genero"]
tipos_pago = datos["tipos_pago"]
descuento_aplicado = datos["descuento_aplicado"]
motivos_devolucion = datos["motivos_devolucion"]
estados_devolucion = datos["estados_devolucion"]
calificacion_proveedores = datos["calificacion_proveedores"]
activos = datos["activos"]
anomalias = datos["anomalias"]
catalogo_por_categoria = datos["catalogo_por_categoria"]

# Fechas
fecha_inicio = datos["rango_fechas"]["inicio"]
fecha_fin = datos["rango_fechas"]["fin"]

# Volúmenes
volumenes = datos["volumenes"]

porcentaje_nulos = datos["valores_nulos"]
formatos_salida = datos["formatos_salida"]

### Conversión de fechas a formato `datetime`

Las variables `fecha_inicio` y `fecha_fin`, extraídas de la configuración como cadenas de texto (`str`) en formato `"YYYY-MM-DD"`, se convierten a objetos `datetime` mediante `datetime.strptime`.

Esta conversión es necesaria para poder realizar operaciones de fecha (comparaciones, cálculo de rangos, generación de fechas aleatorias, etc.) en los pasos posteriores del proceso de simulación de datos.

In [41]:
fecha_inicio = datetime.strptime(fecha_inicio, "%Y-%m-%d")
fecha_fin = datetime.strptime(fecha_fin, "%Y-%m-%d")

## Funciones auxiliares para la generación de datos

Con el fin de mantener un código modular, reutilizable y fácil de mantener, se implementó un conjunto de funciones auxiliares encargadas de generar los distintos atributos utilizados en las tablas del proyecto. Cada función obtiene la información necesaria desde el archivo de configuración (`config.yaml`) y genera valores sintéticos respetando las distribuciones y restricciones definidas para RetailMax.

---

### Generación de fecha y hora de la transacción

Para simular el momento exacto en que ocurre cada venta, se implementaron tres funciones auxiliares que generan la fecha, seleccionan una franja horaria y calculan una hora específica dentro de esa franja. Este enfoque permite obtener distribuciones temporales coherentes con el comportamiento esperado de un entorno de retail, en lugar de una distribución uniforme durante todo el día.

#### `generar_fecha(fecha_inicio, fecha_fin)`

Genera una fecha aleatoria dentro del rango definido en el archivo de configuración utilizando `fake.date_between()`. Esto garantiza que todas las transacciones se encuentren dentro del período histórico establecido para la simulación.

#### `generar_franja(franjas_horarias)`

Selecciona una franja horaria mediante una distribución ponderada, respetando las probabilidades configuradas para cada intervalo del día.

#### `generar_hora_en_franja(franja)`

Genera una hora aleatoria dentro del intervalo correspondiente a la franja seleccionada:

1. Convierte las horas de inicio y fin a objetos `datetime`.
2. Calcula el intervalo disponible mediante `timedelta`.
3. Genera un desplazamiento aleatorio en segundos.
4. Obtiene la hora final sumando dicho desplazamiento a la hora de inicio.

#### Flujo de generación

```text
fecha_inicio, fecha_fin
        │
        ▼
  generar_fecha()
        │
        ▼
 generar_franja()
        │
        ▼
generar_hora_en_franja()
        │
        ▼
Fecha y hora de la transacción
```

---

### Generación de información demográfica

Las siguientes funciones permiten generar la información básica de los miembros del programa de fidelización.

#### `generar_edad(edades)`

Genera una edad utilizando una distribución normal truncada, respetando la media, la desviación estándar y los límites mínimo y máximo definidos en la configuración.

#### `generar_rango_edades(edad, rango_edad_bins)`

Clasifica la edad generada dentro del rango correspondiente según la configuración (`18-25`, `26-35`, `36-45`, etc.).

#### `generar_genero(genero)`

Selecciona un género mediante una distribución ponderada utilizando las probabilidades definidas en el archivo de configuración.

---

### Generación de ubicación geográfica

Estas funciones permiten asignar un país y una ciudad coherentes para cada registro.

#### `generar_pais(paises)`

Selecciona un país utilizando las probabilidades definidas para cada uno.

#### `generar_ciudad(pais, paises)`

Una vez seleccionado el país, genera aleatoriamente una ciudad perteneciente a dicho país.

---

### Generación de productos

Las siguientes funciones generan la información relacionada con los artículos comercializados.

#### `generar_categoria(macro_categorias)`

Selecciona una categoría de producto respetando la distribución configurada.

#### `generar_precio(categoria, rango_precios_por_categoria)`

Genera un precio aleatorio dentro del rango permitido para la categoría seleccionada.

---

### Generación de tiendas y canales

#### `generar_tipo_tienda(tipos_tienda)`

Selecciona el tipo de tienda mediante una distribución ponderada.

#### `generar_canal(canales)`

Genera el canal por el cual se realiza la venta (tienda física, e-commerce o marketplace), respetando las probabilidades definidas.

---

### Generación de información comercial

#### `generar_tipo_pago(tipos_pago)`

Selecciona el método de pago de acuerdo con la distribución configurada.

#### `generar_descuento_aplicado(descuento_aplicado)`

Determina si una venta recibe descuento. En caso afirmativo, genera un porcentaje aleatorio dentro del rango establecido; de lo contrario, retorna un descuento de `0.0`.

---

### Generación de proveedores

#### `generar_calificacion_proveedores(calificacion_proveedores)`

Genera una calificación utilizando una distribución normal truncada, garantizando valores comprendidos entre el mínimo y el máximo configurados.

---

### Generación del estado de registros

#### `generar_activo(activos, tipo)`

Determina si un registro se encuentra activo o inactivo, utilizando la probabilidad correspondiente al tipo de entidad (artículos, proveedores, tiendas o miembros).

---

### Generación de devoluciones

#### `generar_estado_devolucion(estados_devolucion)`

Selecciona aleatoriamente el estado de una devolución según la distribución configurada.

#### `generar_motivo_devolucion(motivos_devolucion)`

Selecciona un motivo de devolución respetando las probabilidades definidas y devuelve toda la información asociada al motivo.

---

### Generación de centros de distribución

#### `generar_centro_distribucion(pais, centros_distribucion)`

Obtiene un centro de distribución compatible con el país previamente generado, garantizando coherencia geográfica en los datos.

---

### Generación de inventario

Para simular el comportamiento del inventario se implementaron tres funciones complementarias.

#### `generar_stock_minimo(stock)`

Genera el nivel mínimo de inventario permitido dentro del rango configurado.

#### `generar_stock_maximo(stock, stock_minimo)`

Genera el nivel máximo de inventario asegurando que siempre sea mayor o igual al stock mínimo.

#### `generar_stock_fisico(stock, stock_minimo, stock_maximo)`

Calcula el stock físico aplicando una variación porcentual sobre los niveles mínimo y máximo definidos en la configuración, obteniendo un valor coherente para la simulación diaria.

---

En conjunto, estas funciones constituyen la base del generador de datos sintéticos del proyecto, permitiendo construir registros consistentes y reproducibles a partir de las reglas de negocio establecidas en el archivo `config.yaml`.

In [42]:
def generar_fecha(fecha_inicio, fecha_fin):
    fecha_aleatoria = fake.date_between(start_date=fecha_inicio, end_date=fecha_fin)
    return fecha_aleatoria


def generar_franja(franjas_horarias):
    probabilidades = [franja["probabilidad"] for franja in franjas_horarias]
    franja_aleatoria = np.random.choice(franjas_horarias, p=probabilidades)
    return franja_aleatoria


def generar_hora_en_franja(franja):
    inicio = datetime.strptime(franja["inicio"], "%H:%M")
    fin = datetime.strptime(franja["fin"], "%H:%M")
    delta = fin - inicio
    segundos_aleatorios = random.randint(0, int(delta.total_seconds()))
    hora_aleatoria = inicio + timedelta(seconds=segundos_aleatorios)
    return hora_aleatoria.time()


def generar_edad(edades):
    # Definir límites de edad deseados y parámetros
    media = edades["media"]
    desviacion = edades["desviacion"]
    edad_min = edades["minimo"]
    edad_max = edades["maximo"]
    # Estandarizar los límites
    a, b = (edad_min - media) / desviacion, (edad_max - media) / desviacion
    # Generar datos truncados
    edad_trunc = truncnorm.rvs(a, b, loc=media, scale=desviacion)
    edad_final = int(np.round(edad_trunc))
    return edad_final


def generar_rango_edades(edad, rango_edad_bins):
    for rango in rango_edad_bins:
        if rango["min"] <= edad <= rango["max"]:
            return rango["rango"]
    return None


def generar_genero(genero):
    nombres = [g["nombre"] for g in genero]
    probabilidades = [g["probabilidad"] for g in genero]
    genero_aleatorio = np.random.choice(nombres, p=probabilidades)
    return genero_aleatorio


def generar_pais(paises):
    nombres = list(paises.keys())
    probabilidades = [p["probabilidad"] for p in paises.values()]

    pais_aleatorio = np.random.choice(nombres, p=probabilidades)
    return pais_aleatorio


def generar_ciudad(pais, paises):
    ciudades = paises[pais]["ciudades"]
    ciudad_aleatoria = np.random.choice(ciudades)
    return ciudad_aleatoria


def generar_categoria(macro_categorias):
    categorias = [cat["nombre"] for cat in macro_categorias]
    probabilidades = [cat["probabilidad"] for cat in macro_categorias]
    categoria_aleatoria = np.random.choice(categorias, p=probabilidades)
    return categoria_aleatoria

def generar_tipo_tienda(tipos_tienda):
    nombres = [t["nombre"] for t in tipos_tienda]
    probabilidades = [t["probabilidad"] for t in tipos_tienda]
    tipo_tienda_aleatorio = np.random.choice(nombres, p=probabilidades)
    return tipo_tienda_aleatorio

def generar_canal(canales):
    nombres = [c["nombre"] for c in canales]
    probabilidades = [c["probabilidad"] for c in canales]
    canal_aleatorio = np.random.choice(nombres, p=probabilidades)
    return canal_aleatorio

def generar_tipo_pago(tipos_pago):
    nombres = [t["nombre"] for t in tipos_pago]
    probabilidades = [t["probabilidad"] for t in tipos_pago]
    tipo_pago_aleatorio = np.random.choice(nombres, p=probabilidades)
    return tipo_pago_aleatorio

def generar_descuento_aplicado(descuento_aplicado):
    probabilidad = descuento_aplicado['probabilidad_con_descuento']
    if np.random.random() < probabilidad:
        minimo = descuento_aplicado["rango_porcentaje"]["min"]
        maximo = descuento_aplicado["rango_porcentaje"]["max"]
        return round(np.random.uniform(minimo, maximo), 2)
    return 0.0

def generar_precio(categoria, rango_precios_por_categoria):
    precio_min = rango_precios_por_categoria[categoria]["min"]
    precio_max = rango_precios_por_categoria[categoria]["max"]
    precio = np.random.uniform(precio_min, precio_max)
    return round(precio, 2)

def generar_calificacion_proveedores(calificacion_proveedores):
    media = calificacion_proveedores["media"]
    desviacion = calificacion_proveedores["desviacion"]
    calificacion_min = calificacion_proveedores["minimo"]
    calificacion_max = calificacion_proveedores["maximo"]

    a = (calificacion_min - media) / desviacion
    b = (calificacion_max - media) / desviacion

    calificacion_trunc = truncnorm.rvs(
        a, b,
        loc=media,
        scale=desviacion
    )

    calificacion_final = int(np.round(calificacion_trunc))
    return calificacion_final

def generar_activo(activos, tipo):
    probabilidad = activos[tipo]
    return np.random.random() < probabilidad

def generar_estado_devolucion(estados_devolucion):
    nombre = [dev["nombre"] for dev in estados_devolucion]
    probabilidades = [dev["probabilidad"] for dev in estados_devolucion]
    estados_devolucion_aleatorio = np.random.choice(nombre, p=probabilidades)
    return estados_devolucion_aleatorio

def generar_motivo_devolucion(motivos_devolucion):
    probabilidades = [m["probabilidad"] for m in motivos_devolucion]
    motivo = np.random.choice(motivos_devolucion, p=probabilidades)
    return motivo

def generar_centro_distribucion(pais, centros_distribucion):
    centros = [
        centro for centro in centros_distribucion if centro["pais_asociado"] == pais
    ]

    return np.random.choice(centros)

def generar_stock_minimo(stock):
    minimo = stock["stock_minimo_config"]["min"]
    maximo = stock["stock_minimo_config"]["max"]

    return random.randint(minimo, maximo)

def generar_stock_maximo(stock, stock_minimo):
    maximo_config = stock["stock_maximo_config"]["max"]

    return random.randint(stock_minimo, maximo_config)

def generar_stock_fisico(stock, stock_minimo, stock_maximo):
    variacion = stock["stock_fisico_variacion"]

    limite_inferior = int(stock_minimo * (1 - variacion))
    limite_superior = int(stock_maximo * (1 + variacion))

    stock_fisico = random.randint(
        max(0, limite_inferior),
        limite_superior
    )

    return stock_fisico

## Funciones utilitarias generales

Antes de definir las funciones que utilizan las distribuciones configuradas en `config.yaml`, se implementan un conjunto de funciones utilitarias encargadas de generar identificadores y atributos sintéticos reutilizables en las diferentes tablas del proyecto. Estas funciones no dependen de distribuciones estadísticas, sino que generan valores con formatos específicos para simular información propia de un sistema de retail.

#### `generar_id(prefijo)`

Genera un identificador único utilizando `uuid.uuid4()`. Se toman los primeros ocho caracteres del identificador, se convierten a mayúsculas y se agrega el prefijo recibido como parámetro para identificar la entidad correspondiente (por ejemplo, `ART`, `PRO`, `TDA` o `CLI`).

#### `generar_nombre_producto(categoria, catalogo_por_categoria)`

Genera el nombre de un producto seleccionando aleatoriamente un artículo del catálogo asociado a la categoría y concatenándolo con una marca ficticia generada mediante `Faker`. De esta manera, se obtienen nombres variados y coherentes con la categoría del producto.

#### `generar_sku(categoria)`

Genera un código SKU tomando las iniciales de las palabras que componen la categoría (ignorando palabras de dos caracteres o menos) y agregando un número aleatorio de cinco dígitos. Esto permite obtener códigos fáciles de identificar y relacionados con la categoría del producto.

#### `generar_codigo_barras()`

Genera un código de barras sintético en formato **EAN-13** utilizando la función `fake.ean13()` de `Faker`, obteniendo códigos con una estructura válida sin necesidad de implementar manualmente el cálculo del dígito verificador.

#### `generar_email()`

Genera una dirección de correo electrónico sintética mediante `fake.email()`, produciendo direcciones con un formato válido que pueden utilizarse para clientes, proveedores u otras entidades del sistema.

#### `generar_telefono()`

Genera un número telefónico sintético utilizando `fake.phone_number()`. Dado que el proyecto emplea la configuración regional `es_CO`, los números generados siguen el formato correspondiente a Colombia.

---

In [43]:
def generar_id(prefijo):
    identificador = uuid.uuid4().hex[:8].upper()
    return f"{prefijo}_{identificador}"

def generar_nombre_producto(categoria, catalogo_por_categoria):
    productos = catalogo_por_categoria[categoria]
    producto = np.random.choice(productos)
    marca = fake.company().split()[0]
    return f"{producto} {marca}"

def generar_sku(categoria):
    palabras = categoria.upper().split()
    prefijo = "".join(
        palabra[0]
        for palabra in palabras
        if len(palabra) > 2
    )[:3]
    numero = random.randint(10000, 99999)
    return f"{prefijo}-{numero}"

def generar_codigo_barras():
    return fake.ean13()

def generar_email():
    return fake.email()

def generar_telefono():
    return fake.phone_number()

id_articulo = generar_id("ART")
nombre_producto = generar_nombre_producto("Alimentos y bebidas", catalogo_por_categoria)
sku = generar_sku("Alimentos y bebidas")
codigo_barras = generar_codigo_barras()
email = generar_email()
telefono = generar_telefono()

### Generación de artículos (`MSTR_ARTICULOS`)

Construye un registro completo para la tabla `MSTR_ARTICULOS`, combinando las funciones auxiliares de distribución (categoría, nombre, SKU, código de barras, precio y estado) para producir un artículo coherente con las reglas configuradas en `config.yaml`.

**Campos generados:**

| Campo | Función utilizada | Descripción |
|---|---|---|
| `id_articulo` | `generar_id("ART")` | Identificador único con prefijo `ART`. |
| `categoria` | `generar_categoria(macro_categorias)` | Macro categoría, según distribución ponderada. |
| `nombre_producto` | `generar_nombre_producto(categoria, catalogo_por_categoria)` | Nombre del producto tomado del catálogo de la categoría + marca ficticia. |
| `sku` | `generar_sku(categoria)` | Código SKU derivado de las iniciales de la categoría. |
| `codigo_barras` | `generar_codigo_barras()` | Código de barras sintético en formato EAN-13. |
| `precio` | `generar_precio(categoria, rango_precios_por_categoria)` | Precio dentro del rango definido para la categoría. |
| `activo` | `generar_activo(activos, "articulos")` | Estado del artículo (activo/inactivo), según probabilidad configurada. |

**Flujo de generación**

```text
generar_categoria()
        │
        ▼
generar_nombre_producto() ──► generar_sku()
        │
        ▼
generar_codigo_barras()
        │
        ▼
  generar_precio()
        │
        ▼
  generar_activo()
        │
        ▼
  Registro MSTR_ARTICULOS
```

> La `categoria` se genera una sola vez y se reutiliza como entrada para `nombre_producto`, `sku` y `precio`, garantizando que estos campos sean coherentes entre sí dentro del mismo registro.

In [44]:
def MSTR_ARTICULOS():
    id_articulo = generar_id("ART")

    categoria = generar_categoria(macro_categorias)

    nombre_producto = generar_nombre_producto(
        categoria,
        catalogo_por_categoria
    )

    sku = generar_sku(categoria)

    codigo_barras = generar_codigo_barras()

    precio = generar_precio(
        categoria,
        rango_precios_por_categoria
    )

    activo = generar_activo(activos, "articulos")

    return {
        "id_articulo": id_articulo,
        "nombre_producto": nombre_producto,
        "categoria": categoria,
        "sku": sku,
        "codigo_barras": codigo_barras,
        "precio": precio,
        "activo": activo
    }

### Generación de proveedores (`MSTR_PROVEEDORES`)

Construye un registro completo para la tabla `MSTR_PROVEEDORES`, combinando datos generados con `Faker` (nombre, correo, teléfono) con las funciones auxiliares de distribución (ubicación, centro de distribución, calificación y estado) definidas en `config.yaml`.

**Campos generados:**

| Campo | Función utilizada | Descripción |
|---|---|---|
| `id_proveedor` | `generar_id("PRO")` | Identificador único con prefijo `PRO`. |
| `nombre_proveedor` | `fake.company()` | Nombre de empresa sintético generado con Faker. |
| `pais` | `generar_pais(paises)` | País del proveedor, según distribución ponderada. |
| `ciudad` | `generar_ciudad(pais, paises)` | Ciudad coherente con el país seleccionado. |
| `centro_distribucion` | `generar_centro_distribucion(pais, centros_distribucion)` | Centro de distribución asociado al país del proveedor. |
| `email` | `generar_email()` | Correo electrónico sintético. |
| `telefono` | `generar_telefono()` | Número telefónico sintético en formato colombiano (`es_CO`). |
| `calificacion` | `generar_calificacion_proveedores(calificacion_proveedores)` | Calificación mediante distribución normal truncada. |
| `activo` | `generar_activo(activos, "proveedores")` | Estado del proveedor (activo/inactivo), según probabilidad configurada. |

**Flujo de generación**

```text
generar_pais()
     │
     ▼
generar_ciudad() ──► generar_centro_distribucion()
     │
     ▼
generar_email() / generar_telefono()
     │
     ▼
generar_calificacion_proveedores()
     │
     ▼
   generar_activo()
     │
     ▼
Registro MSTR_PROVEEDORES
```

> El `pais` se genera una sola vez y se reutiliza para `ciudad` y `centro_distribucion`, garantizando coherencia geográfica dentro del mismo registro.

In [45]:
def MSTR_PROVEEDORES():

    id_proveedor = generar_id("PRO")

    nombre_proveedor = fake.company()

    pais = generar_pais(paises)

    ciudad = generar_ciudad(pais, paises)

    centro_distribucion = generar_centro_distribucion(
        pais,
        centros_distribucion
    )

    email = generar_email()

    telefono = generar_telefono()

    calificacion = generar_calificacion_proveedores(
        calificacion_proveedores
    )

    activo = generar_activo(
        activos,
        "proveedores"
    )

    return {
        "id_proveedor": id_proveedor,
        "nombre_proveedor": nombre_proveedor,
        "pais": pais,
        "ciudad": ciudad,
        "centro_distribucion": centro_distribucion["nombre"],
        "email": email,
        "telefono": telefono,
        "calificacion": calificacion,
        "activo": activo
    }

### Generación de tiendas (`MSTR_TIENDAS`)

Construye un registro completo para la tabla `MSTR_TIENDAS`, combinando las funciones auxiliares de distribución (tipo de tienda, ubicación, centro de distribución y estado) definidas en `config.yaml`.

**Campos generados:**

| Campo | Función utilizada | Descripción |
|---|---|---|
| `id_tienda` | `generar_id("TDA")` | Identificador único con prefijo `TDA`. |
| `tipo_tienda` | `generar_tipo_tienda(tipos_tienda)` | Tipo de tienda, según distribución ponderada. |
| `pais` | `generar_pais(paises)` | País de la tienda, según distribución ponderada. |
| `ciudad` | `generar_ciudad(pais, paises)` | Ciudad coherente con el país seleccionado. |
| `centro_distribucion` | `generar_centro_distribucion(pais, centros_distribucion)` | Centro de distribución asociado al país de la tienda. |
| `activa` | `generar_activo(activos, "tiendas")` | Estado de la tienda (activa/inactiva), según probabilidad configurada. |

**Flujo de generación**

```text
generar_tipo_tienda()
        │
        ▼
   generar_pais()
        │
        ▼
generar_ciudad() ──► generar_centro_distribucion()
        │
        ▼
   generar_activo()
        │
        ▼
Registro MSTR_TIENDAS
```

> El `pais` se genera una sola vez y se reutiliza para `ciudad` y `centro_distribucion`, garantizando coherencia geográfica dentro del mismo registro.

In [46]:
def MSTR_TIENDAS():

    id_tienda = generar_id("TDA")

    tipo_tienda = generar_tipo_tienda(tipos_tienda)

    pais = generar_pais(paises)

    ciudad = generar_ciudad(pais, paises)

    centro_distribucion = generar_centro_distribucion(
        pais,
        centros_distribucion
    )

    activa = generar_activo(
        activos,
        "tiendas"
    )

    return {
        "id_tienda": id_tienda,
        "tipo_tienda": tipo_tienda,
        "pais": pais,
        "ciudad": ciudad,
        "centro_distribucion": centro_distribucion["nombre"],
        "activa": activa
    }

### Generación de miembros (`CRM_MIEMBROS`)

Construye un registro completo para la tabla `CRM_MIEMBROS`, combinando las funciones auxiliares de distribución (edad, género, ubicación y estado) definidas en `config.yaml` para simular el perfil demográfico de los clientes del programa de fidelización.

**Campos generados:**

| Campo | Función utilizada | Descripción |
|---|---|---|
| `id_miembro` | `generar_id("CLI")` | Identificador único con prefijo `CLI`. |
| `edad` | `generar_edad(edades)` | Edad mediante distribución normal truncada (media, desviación, mínimo y máximo configurados). |
| `rango_edad` | `generar_rango_edades(edad, rango_edad_bins)` | Clasificación de la edad en el bin correspondiente (`18-25`, `26-35`, etc.). |
| `genero` | `generar_genero(genero)` | Género, según distribución ponderada. |
| `pais` | `generar_pais(paises)` | País del miembro, según distribución ponderada. |
| `ciudad` | `generar_ciudad(pais, paises)` | Ciudad coherente con el país seleccionado. |
| `activo` | `generar_activo(activos, "miembros")` | Estado del miembro (activo/inactivo), según probabilidad configurada. |

**Flujo de generación**

```text
generar_edad() ──► generar_rango_edades()
        │
        ▼
   generar_genero()
        │
        ▼
generar_pais() ──► generar_ciudad()
        │
        ▼
   generar_activo()
        │
        ▼
Registro CRM_MIEMBROS
```

> La `edad` se genera una sola vez y se reutiliza para calcular `rango_edad`, y el `pais` se reutiliza para `ciudad`, garantizando coherencia entre estos campos dentro del mismo registro.

In [47]:
def CRM_MIEMBROS():

    id_miembro = generar_id("CLI")

    edad = generar_edad(edades)

    rango_edad = generar_rango_edades(
        edad,
        rango_edad_bins
    )

    genero_salida = generar_genero(genero)

    pais = generar_pais(paises)

    ciudad = generar_ciudad(pais, paises)

    activo = generar_activo(
        activos,
        "miembros"
    )

    return {
        "id_miembro": id_miembro,
        "edad": edad,
        "rango_edad": rango_edad,
        "genero": genero_salida,
        "pais": pais,
        "ciudad": ciudad,
        "activo": activo
    }

### Generación de ventas (`FACT_VENTAS`)

Construye un registro completo para la tabla de hechos `TRANS_VENTAS`, combinando la generación de fecha/hora en horario pico, el muestreo de dimensiones relacionadas (`cliente`, `articulo`, `tienda`) y las funciones auxiliares de distribución (canal, tipo de pago y descuento) definidas en `config.yaml`.

**Campos generados:**

| Campo | Función / origen | Descripción |
|---|---|---|
| `id_venta` | `generar_id("VTA")` | Identificador único con prefijo `VTA`. |
| `fecha` | `generar_fecha(fecha_inicio, fecha_fin)` | Fecha aleatoria dentro del rango histórico configurado. |
| `hora` | `generar_franja()` + `generar_hora_en_franja()` | Hora dentro de una franja horaria ponderada (simula horarios pico). |
| `id_cliente` | `df_miembros.sample()` | Cliente tomado aleatoriamente de `MSTR_CRM_MIEMBROS`. |
| `id_articulo` | `df_articulos.sample(weights=pesos_articulos)` | Artículo muestreado con pesos de Pareto, concentrando ventas en pocos productos. |
| `id_tienda` | `df_tiendas.sample(weights=pesos_tiendas)` | Tienda muestreada con pesos de Pareto, concentrando ventas en pocas tiendas. |
| `canal` | `generar_canal(canales)` | Canal de venta (física, e-commerce, marketplace), según distribución ponderada. |
| `tipo_pago` | `generar_tipo_pago(tipos_pago)` | Método de pago, según distribución configurada. |
| `descuento` | `generar_descuento_aplicado(descuento_aplicado)` | Porcentaje de descuento aplicado, o `0.0` si no aplica. |
| `precio` | `articulo["precio"]` | Precio tomado directamente del artículo seleccionado. |

**Flujo de generación**

```text
generar_fecha()
      │
      ▼
generar_franja() ──► generar_hora_en_franja()
      │
      ▼
df_miembros.sample()
      │
      ▼
df_articulos.sample(weights=pesos_articulos) ──► precio
      │
      ▼
df_tiendas.sample(weights=pesos_tiendas)
      │
      ▼
generar_canal() / generar_tipo_pago() / generar_descuento_aplicado()
      │
      ▼
Registro FACT_VENTAS
```

> `articulo` y `tienda` se muestrean usando `pesos_articulos` y `pesos_tiendas` (distribución de Pareto calculada previamente), en lugar de un muestreo uniforme, para reflejar el principio 80/20 en el comportamiento de ventas: unos pocos artículos y tiendas concentran la mayoría de las transacciones. El campo `precio` se toma del mismo `articulo` seleccionado, garantizando coherencia entre ambos campos.

In [48]:
def FACT_VENTAS():
    id_venta = generar_id("VTA")

    fecha = generar_fecha(
        fecha_inicio,
        fecha_fin
    )

    franja = generar_franja(
        franjas_horarias
    )

    hora = generar_hora_en_franja(
        franja
    )

    cliente = random.choice(clientes)

    articulo = random.choice(articulos)

    tienda = random.choice(tiendas)

    canal = generar_canal(
        canales
    )

    tipo_pago = generar_tipo_pago(
        tipos_pago
    )

    descuento = generar_descuento_aplicado(
        descuento_aplicado
    )

    precio = articulo["precio"]

    return {
        "id_venta": id_venta,
        "fecha": fecha,
        "hora": hora,
        "id_cliente": cliente["id_miembro"],
        "id_articulo": articulo["id_articulo"],
        "id_tienda": tienda["id_tienda"],
        "canal": canal,
        "tipo_pago": tipo_pago,
        "descuento": descuento,
        "precio": precio
    }

### Generación de inventario diario (`INV_STOCK_DIARIO`)

Construye un registro completo para la tabla `INV_STOCK_DIARIO`, combinando el muestreo de dimensiones relacionadas (`articulo`, `tienda`) con las funciones auxiliares que calculan los niveles de stock mínimo, máximo y físico definidos en `config.yaml`.

**Campos generados:**

| Campo | Función / origen | Descripción |
|---|---|---|
| `id_articulo` | `df_articulos.sample()` | Artículo tomado aleatoriamente de `MSTR_ARTICULOS`. |
| `id_tienda` | `df_tiendas.sample()` | Tienda tomada aleatoriamente de `MSTR_TIENDAS`. |
| `stock_minimo` | `generar_stock_minimo(stock)` | Nivel mínimo de inventario dentro del rango configurado. |
| `stock_maximo` | `generar_stock_maximo(stock, stock_minimo)` | Nivel máximo de inventario, siempre mayor o igual al mínimo. |
| `stock_fisico` | `generar_stock_fisico(stock, stock_minimo, stock_maximo)` | Stock físico real, aplicando una variación porcentual sobre los niveles mínimo y máximo. |

**Flujo de generación**

```text
df_articulos.sample()
        │
        ▼
df_tiendas.sample()
        │
        ▼
generar_stock_minimo()
        │
        ▼
generar_stock_maximo() ──► usa stock_minimo
        │
        ▼
generar_stock_fisico() ──► usa stock_minimo y stock_maximo
        │
        ▼
Registro INV_STOCK_DIARIO
```

> `stock_minimo` se genera primero y se reutiliza como entrada para `stock_maximo`, y ambos se reutilizan para calcular `stock_fisico`, garantizando que los tres niveles sean coherentes entre sí dentro del mismo registro.

In [49]:
def INV_STOCK_DIARIO():
    articulo = df_articulos.sample().iloc[0]

    tienda = df_tiendas.sample().iloc[0]

    stock_minimo = generar_stock_minimo(stock)

    stock_maximo = generar_stock_maximo(
        stock,
        stock_minimo
    )

    stock_fisico = generar_stock_fisico(
        stock,
        stock_minimo,
        stock_maximo
    )

    return {
        "id_articulo": articulo["id_articulo"],
        "id_tienda": tienda["id_tienda"],
        "stock_minimo": stock_minimo,
        "stock_maximo": stock_maximo,
        "stock_fisico": stock_fisico
    }

### Generación de devoluciones (`FACT_DEVOLUCIONES`)

Construye un registro completo para la tabla `POST_DEVOLUCIONES`, combinando el muestreo de una venta existente con las funciones auxiliares de distribución que determinan el estado y el motivo de la devolución, definidos en `config.yaml`.

**Campos generados:**

| Campo | Función / origen | Descripción |
|---|---|---|
| `id_venta` | `df_ventas.sample()` | Venta tomada aleatoriamente de `TRANS_VENTAS`, sobre la cual se registra la devolución. |
| `estado` | `generar_estado_devolucion(estados_devolucion)` | Estado de la devolución, según distribución ponderada. |
| `motivo` | `generar_motivo_devolucion(motivos_devolucion)` | Motivo de la devolución, según distribución ponderada (se toma el campo `descripcion`). |

**Flujo de generación**

```text
df_ventas.sample()
        │
        ▼
generar_estado_devolucion()
        │
        ▼
generar_motivo_devolucion()
        │
        ▼
Registro FACT_DEVOLUCIONES
```

> Al depender de `df_ventas`, esta función debe ejecutarse **después** de generar `df_ventas`, garantizando integridad referencial entre `POST_DEVOLUCIONES` y `TRANS_VENTAS`.

In [50]:
def FACT_DEVOLUCIONES():
    venta = random.choice(ventas_lista)

    estado = generar_estado_devolucion(
        estados_devolucion
    )

    motivo = generar_motivo_devolucion(
        motivos_devolucion
    )

    return {
        "id_venta": venta["id_venta"],
        "estado": estado,
        "motivo": motivo["descripcion"]
    }

In [51]:
print(volumenes)

{'articulos': 5000, 'proveedores': 800, 'tiendas': 150, 'miembros': 50000, 'ventas': 1000000, 'inventario': 750000, 'devoluciones': 50000}


In [ ]:
articulos = []
for i in range(volumenes["articulos"]):
    articulos.append(MSTR_ARTICULOS())
df_articulos = pd.DataFrame(articulos)

proveedores = []
for i in range(volumenes["proveedores"]):
    proveedores.append(MSTR_PROVEEDORES())
df_proveedores = pd.DataFrame(proveedores)

tiendas = []
for i in range(volumenes["tiendas"]):
    tiendas.append(MSTR_TIENDAS())
df_tiendas = pd.DataFrame(tiendas)

miembros = []
for i in range(volumenes["miembros"]):
    miembros.append(CRM_MIEMBROS())
df_miembros = pd.DataFrame(miembros)

clientes = df_miembros.to_dict("records")
articulos = df_articulos.to_dict("records")
tiendas = df_tiendas.to_dict("records")

ventas = []
for i in range(volumenes["ventas"]):
    ventas.append(FACT_VENTAS())
df_ventas = pd.DataFrame(ventas)

stock_diario = []
for i in range(volumenes["inventario"]):
    stock_diario.append(INV_STOCK_DIARIO())
df_stock_diario = pd.DataFrame(stock_diario)

ventas_lista = df_ventas.to_dict("records")
devoluciones = []
for i in range(volumenes["devoluciones"]):
    devoluciones.append(FACT_DEVOLUCIONES())
df_devoluciones = pd.DataFrame(devoluciones)

1
2
3
4
1000000
Este no da
5
Inicio devoluciones
0
5000
10000
15000
20000
25000
30000
35000
40000
45000
Terminó el for
6


In [55]:
(df_ventas.isnull().mean() * 100).round(2)

id_venta       0.0
fecha          0.0
hora           0.0
id_cliente     0.0
id_articulo    0.0
id_tienda      0.0
canal          0.0
tipo_pago      0.0
descuento      0.0
precio         0.0
dtype: float64

Guardar todas las lebrerias con las versiones instaladas funcionales para el proyecto

In [53]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
